# Segmentacion Inteligente de Clientes Empresariales

**Algoritmo**: K-Means con seleccion de K via Elbow + Silhouette
**Objetivo**: Descubrir arquetipos de cliente para acciones comerciales diferenciales

> El target no existe antes del modelo: lo genera el propio algoritmo.
> La interpretacion de cada cluster es trabajo del analista, no del algoritmo.

**Lo que aprenderas:**
1. Por que la normalizacion es CRITICA en K-Means (a diferencia de RandomForest)
2. Como elegir el numero de clusters con criterio cuantitativo
3. Como interpretar los centroides en lenguaje de negocio
4. Cuando el clustering es util y cuando es una ilusion

---

## Contexto del caso de negocio

| | |
|---|---|
| **Empresa** | empresa — área comercial B2B y marketing de cuentas |
| **Problema de negocio** | Agrupar la cartera de clientes empresariales en segmentos homogéneos para diseñar propuestas de valor diferenciadas y asignar recursos comerciales de forma eficiente |
| **Datos disponibles** | 250 clientes B2B con variables de comportamiento: volumen de facturación, número de productos contratados, frecuencia de soporte, antigüedad, sector y tamaño de empresa |
| **Técnica aplicada** | Clustering no supervisado con K-Means; número óptimo de clusters determinado por el método del codo y el coeficiente Silhouette; reducción PCA 2D para visualización |
| **Salida del modelo** | Segmento asignado a cada cliente (0, 1, 2...) con perfil medio por cluster: facturación media, productos contratados, nivel de soporte y antigüedad |
| **Valor operativo** | Permite al equipo comercial adaptar el discurso y la oferta por segmento (upselling en clientes de alto valor, onboarding reforzado en clientes nuevos de bajo uso) y medir la evolución de cada segmento en el tiempo |

In [ ]:
import os, sys
from pathlib import Path

# Configuracion de entorno: ajusta CWD y descarga datos segun el entorno de ejecucion
_BASE_URL = "https://raw.githubusercontent.com/amador2001/ia-datos/main/"
_CSVS = ["clientes_b2b.csv"]

if "google.colab" in sys.modules:
    import urllib.request
    Path("datos").mkdir(exist_ok=True)
    for _csv in _CSVS:
        _dest = Path("datos") / _csv
        if not _dest.exists():
            urllib.request.urlretrieve(_BASE_URL + _csv, str(_dest))
            print(f"Descargado: {_csv}")
elif "__vsc_ipynb_file__" in dir():
    os.chdir(Path(__vsc_ipynb_file__).parent)
elif not Path("datos").exists():
    for _p in [Path("Jupyter_notebooks"), Path("../Jupyter_notebooks")]:
        if (_p / "datos").exists():
            os.chdir(_p)
            break

print(f"Entorno listo. CWD: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.dpi"] = 110

In [ ]:
# Cargar dataset de clientes B2B
df = pd.read_csv("datos/clientes_b2b.csv")

print("Primeras filas del dataset:")
display(df.head(10))

print("\nEstadisticas descriptivas (variables numericas):")
display(df.describe().round(2))

df.info()

print(f"\nClientes totales: {len(df)}")
print(f"Paises: {df['pais'].unique()}")
print(f"Sectores: {df['sector'].unique()}")
print(f"\nSegmentos reales en los datos (para validacion posterior):")
print(df["segmento"].value_counts())

### Variables del dataset

| Variable | Tipo | Descripcion |
|---|---|---|
| cliente_id | str | Identificador unico |
| facturacion_anual | EUR | Facturacion anual del cliente |
| margen_anual | ratio | Margen bruto [0-1] |
| usuarios_activos_mensual | int | Usuarios que se loguean al menos 1 vez/mes |
| sesiones_semana | float | Media de sesiones semanales |
| modulos_activos | int | Modulos contratados que usa activamente |
| tickets_mes | int | Tickets de soporte abiertos por mes |
| incidencias_criticas | int | Incidencias de nivel critico |
| dias_sin_login | int | Dias desde el ultimo login de cualquier usuario |
| nps | int | Net Promoter Score [-100, +100] |
| antiguedad_meses | int | Meses como cliente |
| pais / sector | str | Variables categoricas |
| segmento | str | Etiqueta real para validacion (no usada en el modelo) |

> **Importante**: el modelo K-Means NO vera la columna `segmento`.
> La usaremos solo al final para validar si el clustering tiene sentido.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

vars_plot = ["facturacion_anual","usuarios_activos_mensual",
             "sesiones_semana","nps","dias_sin_login","tickets_mes"]

colors_seg = {"Champion":"gold","Developing":"steelblue",
              "At-Risk":"tomato","Dormant":"gray"}

for i, var in enumerate(vars_plot):
    for seg, color in colors_seg.items():
        subset = df[df["segmento"]==seg][var]
        subset.plot.kde(ax=axes[i], label=seg, color=color, alpha=0.7)
    axes[i].set_title(f"Distribucion: {var}")
    axes[i].legend(fontsize=7)

plt.suptitle("Distribucion de features por segmento real (referencia)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Variables numericas y categoricas para el modelo
NUM_FEATURES = ["facturacion_anual","margen_anual","usuarios_activos_mensual",
                "sesiones_semana","modulos_activos","tickets_mes",
                "incidencias_criticas","dias_sin_login","nps","antiguedad_meses"]
CAT_FEATURES = ["pais","sector"]

# Preprocesador: escala numericas y codifica categoricas
# K-Means mide distancias euclidianas, por lo que escalar es OBLIGATORIO:
# sin escalar, facturacion_anual (en miles de EUR) domina sobre nps ([-100,+100])
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUM_FEATURES),              # media=0, std=1
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES)  # 0/1 por categoria
])

X = df[NUM_FEATURES + CAT_FEATURES]
X_scaled = preprocessor.fit_transform(X)   # matriz numerica normalizada para K-Means

print(f"Dimensiones del dataset normalizado: {X_scaled.shape}")
print("(filas=clientes, columnas=features numericas + categoricas codificadas)")

# Split 80/20 para validar estabilidad del clustering
from sklearn.model_selection import train_test_split
X_train, X_test = train_test_split(X_scaled, test_size=0.20, random_state=42)
print(f"\nClientes de entrenamiento: {X_train.shape[0]}")
print(f"Clientes de validacion:    {X_test.shape[0]}")

# Dataset de entrenamiento normalizado (primeras 3 filas, primeras 8 columnas)
print("\nPrimeras 3 filas del dataset normalizado (primeras 8 columnas):")
display(pd.DataFrame(X_train[:3, :8],
        columns=NUM_FEATURES[:8]).round(3))

In [ ]:
# ── Metodo del codo (Elbow): buscar el K optimo ───────────────────────────────
# Inertia = suma de distancias al cuadrado de cada punto a su centroide.
# Al aumentar K, la inertia siempre baja. El "codo" es donde la mejora se ralentiza.
inertias, silhouettes = [], []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_train)
    inertias.append(km.inertia_)
    # Silhouette score: mide cohesion interna y separacion entre clusters [-1, +1]
    # Mas alto = clusters mas compactos y separados
    sil = silhouette_score(X_train, km.labels_)
    silhouettes.append(sil)
    print(f"K={k}: inertia={km.inertia_:.0f}, silhouette={sil:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_range), inertias, "bo-")
axes[0].set_title("Metodo del codo (Elbow)")
axes[0].set_xlabel("Numero de clusters K")
axes[0].set_ylabel("Inertia")

axes[1].plot(list(K_range), silhouettes, "go-")
axes[1].set_title("Silhouette Score por K")
axes[1].set_xlabel("Numero de clusters K")
axes[1].set_ylabel("Silhouette Score")
plt.tight_layout()
plt.show()

# ── Entrenar modelo final con K=4 ─────────────────────────────────────────────
K_OPTIMO = 4
kmeans = KMeans(n_clusters=K_OPTIMO, n_init=50, max_iter=500, random_state=42)
kmeans.fit(X_scaled)   # entrenar sobre todos los datos (no solo train)

# Asignar cluster a cada cliente
df["cluster"] = kmeans.labels_
print(f"\nDistribucion de clientes por cluster:")
print(df["cluster"].value_counts().sort_index())

In [ ]:
# predict() asigna cada punto al cluster mas cercano
clusters_test = kmeans.predict(X_test)   # clusters para el 20% de validacion
print(f"Distribucion en test: {pd.Series(clusters_test).value_counts().sort_index().to_dict()}")

# Perfiles de cada cluster en variables originales (sin normalizar)
perfil = df.groupby("cluster")[NUM_FEATURES].mean().round(1)
print("\nPerfil medio de cada cluster:")
display(perfil)

# Cruzar con los segmentos reales para validar la calidad del clustering
print("\nDistribucion de segmentos reales por cluster:")
display(pd.crosstab(df["cluster"], df["segmento"]))

In [ ]:
# Reducir a 2D con PCA para visualizar los clusters
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Nombres de los clusters segun perfil (asignacion manual tras analizar centroides)
nombres = {0:"Champion", 1:"Developing", 2:"At-Risk", 3:"Dormant"}
colors  = ["gold","steelblue","tomato","gray"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PCA 2D con clusters coloreados
for k in range(K_OPTIMO):
    mask = df["cluster"] == k
    axes[0].scatter(X_pca[mask,0], X_pca[mask,1],
                    c=colors[k], label=f"Cluster {k}", alpha=0.6, s=40)
# Centroides en el espacio PCA
centroides_pca = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centroides_pca[:,0], centroides_pca[:,1],
                c="black", marker="*", s=200, label="Centroide")
axes[0].set_title("Clusters K-Means en PCA 2D")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%})")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
axes[0].legend(fontsize=8)

# Heatmap de z-scores de centroides: que define a cada cluster
from sklearn.preprocessing import MinMaxScaler
centros_norm = MinMaxScaler().fit_transform(perfil[NUM_FEATURES[:6]].T).T
im = axes[1].imshow(centros_norm, cmap="RdYlGn", aspect="auto")
axes[1].set_xticks(range(6)); axes[1].set_xticklabels(NUM_FEATURES[:6], rotation=45, ha="right")
axes[1].set_yticks(range(K_OPTIMO)); axes[1].set_yticklabels([f"Cluster {k}" for k in range(K_OPTIMO)])
axes[1].set_title("Perfil normalizado de clusters\n(verde=alto, rojo=bajo)")
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
# ── Clasificar un cliente nuevo en un cluster ─────────────────────────────────
nuevo_cliente = pd.DataFrame([{
    "facturacion_anual":        95000,
    "margen_anual":             0.26,
    "usuarios_activos_mensual": 40,
    "sesiones_semana":          38.0,
    "modulos_activos":          4,
    "tickets_mes":              3,
    "incidencias_criticas":     0,
    "dias_sin_login":           3,
    "nps":                      42,
    "antiguedad_meses":         14,
    "pais":                     "ES",
    "sector":                   "SaaS"
}])

# Normalizar con el mismo preprocessor (CRITICO: transform, no fit_transform)
nuevo_scaled = preprocessor.transform(nuevo_cliente)

# predict() asigna al cluster mas cercano
cluster_asignado = kmeans.predict(nuevo_scaled)[0]

print(f"Cluster asignado: {cluster_asignado}")
print(f"Perfil del cluster {cluster_asignado}:")
display(perfil.loc[[cluster_asignado]])
print(f"\nAccion recomendada segun perfil del cluster:")
acciones = {0:"Cliente premium: foco en expansion y referidos",
            1:"Cliente en crecimiento: formacion y activacion de modulos",
            2:"Cliente en riesgo: contacto proactivo y oferta de retencion",
            3:"Cliente dormido: campana de reactivacion o revision de contrato"}
print(f"  {acciones.get(cluster_asignado, 'Sin accion definida')}")

### Cuando NO usar K-Means

| Situacion | Por que falla |
|---|---|
| Clusters de forma no esferica | K-Means asume clusters esfericos (distancia euclidiana) |
| Numero de clusters desconocido y muy variable | El metodo del codo no siempre da una respuesta clara |
| Variables categoricas sin codificar | Las distancias con texto no tienen sentido |
| Muy pocas muestras (< 50) | Los centroides son inestables |

> **La trampa del clustering**: siempre produce clusters, aunque no existan.
> Validar que los segmentos tienen sentido de negocio antes de actuar sobre ellos.